# Week 1 — Agricultural Data Exploration

## Agricultural Production, Yield & Agribusiness Intelligence — India

**Role:** Junior Data Science Analyst – Agriculture & Agribusiness  
**Project:** Data Exploration and Strategy Formulation  
**Scope:** Selected major Indian crops and official DES normal-estimate periods

### Objective
Explore medium-term patterns in crop area, production and yield, identify important differences across crops, and translate the findings into practical agribusiness questions for deeper analysis.

> **Important:** This notebook is a descriptive Week 1 exploration. It does not establish causal relationships between weather, inputs, prices and crop performance.

## 1. Data Source & Scope

The primary source is the **Directorate of Economics & Statistics (DES), Department of Agriculture & Farmers Welfare, Government of India**. DES publishes Area, Production & Yield statistics and Agricultural Statistics at a Glance reports.

The project dataset is a structured analytical extract from official DES published tables. It contains selected crops across three overlapping five-year normal-estimate periods:

- 2016–17 to 2020–21
- 2019–20 to 2023–24
- 2020–21 to 2024–25

**Variables:**
- `crop` — crop or crop group
- `period` — normal-estimate period
- `area_mha` — area in million hectares
- `production_mt` — production in million tonnes
- `yield_kg_ha` — yield in kg/hectare

Official references:
- DES Agricultural Statistics at a Glance: https://desagri.gov.in/document-report/agricultural-statistics-at-a-glance-2024/
- DES APY reports: https://data.desagri.gov.in/website/crops-report-district-level-web
- India OGD district/season/crop production data: https://www.data.gov.in/resource/district-wise-season-wise-crop-production-statistics-1997

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_PATH = '../data/des_normal_estimates_major_crops.csv'
df = pd.read_csv(DATA_PATH)

df.head()

## 2. Dataset Overview

In [ ]:
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'Unique crops/crop groups: {df["crop"].nunique()}')
print(f'Periods: {df["period"].nunique()}')
        


In [ ]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Unique crops/crop groups', 'Unique periods', 'Missing values', 'Duplicate rows'],
    'Value': [
        len(df),
        df.shape[1],
        df['crop'].nunique(),
        df['period'].nunique(),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
})
summary

## 3. Data Quality Checks

Before interpreting trends, check missing values, duplicates, data types and basic ranges.

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'unique_count': df.nunique(),
})
quality

In [ ]:
numeric_cols = ['area_mha', 'production_mt', 'yield_kg_ha']
range_check = pd.DataFrame({
    'minimum': df[numeric_cols].min(),
    'maximum': df[numeric_cols].max(),
    'mean': df[numeric_cols].mean()
})
range_check

### Quality assessment
The Week 1 extract is intentionally small and structured for strategy validation. No missing values or duplicate observations should be present. A production-grade pipeline should still validate units, crop naming, state/district coverage and revisions when the larger DES/OGD datasets are integrated.

## 4. Descriptive Statistics

In [ ]:
df.groupby('crop')[numeric_cols].agg(['mean', 'min', 'max']).round(2)

## 5. Production Analysis

In [ ]:
production_pivot = df.pivot(index='crop', columns='period', values='production_mt')
production_pivot.sort_values(production_pivot.columns[-1], ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for crop in df['crop'].unique():
    subset = df[df['crop'] == crop]
    plt.plot(subset['period'], subset['production_mt'], marker='o', label=crop)
plt.title('Production by Crop Across DES Normal-Estimate Periods')
plt.xlabel('Normal-estimate period')
plt.ylabel('Production (million tonnes)')
plt.xticks(rotation=20)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Yield / Productivity Analysis

In [ ]:
yield_pivot = df.pivot(index='crop', columns='period', values='yield_kg_ha')
yield_pivot.sort_values(yield_pivot.columns[-1], ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for crop in df['crop'].unique():
    subset = df[df['crop'] == crop]
    plt.plot(subset['period'], subset['yield_kg_ha'], marker='o', label=crop)
plt.title('Yield by Crop Across DES Normal-Estimate Periods')
plt.xlabel('Normal-estimate period')
plt.ylabel('Yield (kg/hectare)')
plt.xticks(rotation=20)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 7. Medium-Term Change Analysis

The comparison below uses the earliest and latest normal-estimate periods. Because the periods overlap, these are **average-period comparisons**, not annual growth rates.

In [ ]:
periods = list(df['period'].drop_duplicates())
first_period, last_period = periods[0], periods[-1]

first = df[df['period'] == first_period].set_index('crop')
last = df[df['period'] == last_period].set_index('crop')

change = pd.DataFrame(index=first.index)
for col in numeric_cols:
    change[f'{col}_pct_change'] = ((last[col] - first[col]) / first[col]) * 100

change.sort_values('production_mt_pct_change', ascending=False).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_data = change['production_mt_pct_change'].sort_values()
ax.barh(plot_data.index, plot_data.values)
ax.set_title(f'Production Change: {first_period} to {last_period}')
ax.set_xlabel('Percentage change in average production (%)')
ax.set_ylabel('Crop / crop group')
plt.tight_layout()
plt.show()

## 8. Key Findings

In [ ]:
foodgrain = df[df['crop'] == 'Total Foodgrains'].sort_values('period')
rice = df[df['crop'] == 'Rice'].sort_values('period')
maize = df[df['crop'] == 'Maize'].sort_values('period')

print(f"Total foodgrain production: {foodgrain.iloc[0]['production_mt']:.2f} Mt -> {foodgrain.iloc[-1]['production_mt']:.2f} Mt")
print(f"Rice production: {rice.iloc[0]['production_mt']:.2f} Mt -> {rice.iloc[-1]['production_mt']:.2f} Mt")
print(f"Maize production: {maize.iloc[0]['production_mt']:.2f} Mt -> {maize.iloc[-1]['production_mt']:.2f} Mt")
print('\nTop production-change observations:')
print(change['production_mt_pct_change'].sort_values(ascending=False).head(5).round(2).to_string())

### Interpretation

1. **Foodgrain production increased materially** between the earliest and latest normal-estimate periods in the analytical extract.
2. **Rice and wheat remain major production contributors**, while maize shows a strong medium-term increase in the selected periods.
3. **Yield is an important productivity lens** because production can change through both cultivated area and yield.
4. Several crops show improving yield alongside production growth, suggesting that future analysis should separate **area expansion** from **productivity improvement**.
5. These results are **descriptive, not causal**. Weather, irrigation, input use, technology, policy and market conditions should be added before making strategic attribution claims.

## 9. Agribusiness Implications

| Analytical signal | Potential business question |
|---|---|
| Production growth | Where is supply capacity expanding? |
| Yield improvement | Which regions/crops may be benefiting from productivity gains? |
| Area expansion | Is growth driven by land allocation rather than yield? |
| Yield gaps | Which regions have room for productivity improvement? |
| Crop diversification | Where could input, storage, processing or advisory demand emerge? |
| Weather sensitivity | Which crop-region combinations are most exposed to rainfall variability? |

**Next analytical layer:** join state/district-level APY data with rainfall, irrigation and market-price/arrival data to identify opportunity and risk hotspots.

## 10. Limitations

- The dataset is a **structured extract**, not the full district-level production database.
- The normal-estimate periods overlap, so they should not be interpreted as independent periods or annual growth rates.
- Production and yield alone cannot explain why performance changed.
- State/district variation is hidden by the national-level aggregation in this extract.
- Market prices, arrivals, rainfall, irrigation and input variables are not yet integrated.
- Official agricultural estimates may be revised; production pipelines should preserve source date and revision status.

## 11. Next Steps for Week 2+

1. Acquire and standardize district/state/year/season/crop APY data from DES/OGD.
2. Add rainfall and rainfall-deviation variables.
3. Add market arrivals and prices where reliable data coverage is available.
4. Build state/crop productivity rankings and yield-gap analysis.
5. Test relationships between rainfall variability and crop performance.
6. Create an agribusiness opportunity/risk scorecard.
7. Develop a dashboard or presentation for decision-makers.

### Success criterion
The final project should move from **descriptive crop statistics → regional diagnosis → explanatory analysis → actionable agribusiness recommendations**.

## 12. Reproducibility

Run the notebook from the `Week-01/notebooks/` directory or open it in Jupyter/VS Code with the repository structure unchanged. The dataset is loaded using a relative path from the notebook to `../data/`.